<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/main/RA1_LAB2/EXPERIENCIA_3%20/RA2_Lab_N%C2%B02_Exp3_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación de Sentimientos en Reseñas de Google Play
## Comparativa de Modelos: Bag of Words vs. Redes Neuronales Secuenciales

---

**Proyecto:** Automatización del análisis de sentimientos para reseñas de aplicaciones móviles
**Dominio:** Procesamiento de Lenguaje Natural (NLP) — Clasificación Multiclase
**Framework:** TensorFlow / Keras + Scikit-Learn
**Autor** :`Acosta Alex`, `Bareiro Santiago`, `Borges Agustin`

---

### Resumen Ejecutivo

Una consultora de análisis de producto enfrenta el problema de escala: la revisión manual de miles de reseñas mensuales de Google Play es inviable operacionalmente. El objetivo de este notebook es diseñar, entrenar y evaluar comparativamente tres arquitecturas de clasificación de texto para mapear cada reseña a una de tres categorías de sentimiento:

| Etiqueta | Rango de Estrellas | Clase Numérica |
|---|---|---|
| **Negativo** | 1 – 2 estrellas | `0` |
| **Neutral**  | 3 estrellas      | `1` |
| **Positivo** | 4 – 5 estrellas  | `2` |

La estrategia experimental sigue el eje **Hipótesis → Experimento → Conclusión**, escalando progresivamente en complejidad: desde un modelo estadístico clásico (TF-IDF + Regresión Logística) hasta arquitecturas neurales profundas que explotan la estructura secuencial del lenguaje (BiLSTM y CNN-1D).

---
## Sección 1 — Configuración del Entorno y Análisis Exploratorio de Datos (EDA)

### 1.1 Importación de Librerías

In [ ]:
# ─── Reproducibilidad ────────────────────────────────────────────────────────
import os, random
import numpy as np
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ─── Manipulación de datos ────────────────────────────────────────────────────
import pandas as pd
import re
from collections import Counter

# ─── Visualización ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec
sns.set_theme(style='whitegrid', palette='muted')

# ─── Scikit-Learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

# ─── TensorFlow / Keras ───────────────────────────────────────────────────────
import tensorflow as tf
tf.random.set_seed(SEED)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Embedding, Dropout, Bidirectional, LSTM,
    Conv1D, GlobalMaxPooling1D, Dense, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")
print("✓ Entorno configurado correctamente.")

# ─── Módulos auxiliares (time/platform/json/shutil/Path) ──
import time, platform, json, shutil
from pathlib import Path

# [Z.3] Carpeta de figuras creada una sola vez, al inicio
Path('figs').mkdir(exist_ok=True)

# Marca de inicio para el tiempo total del notebook
T_INICIO_GLOBAL = time.perf_counter()
print(f"Scikit-Learn version : {__import__('sklearn').__version__}")
print(f"Matplotlib version   : {__import__('matplotlib').__version__}")
print(f"Seaborn version      : {__import__('seaborn').__version__}")
print(f"Semilla global (SEED): {SEED}")


### 1.2 Carga del Dataset

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Carga del dataset — ADAPTADO PARA EJECUCIÓN LOCAL
# Si 'reviews_limpias.csv' ya está en el directorio actual se usa directamente.
# Solo si NO existe se intenta la descarga desde Drive (requiere internet).
# ═══════════════════════════════════════════════════════════════════════════════
import os

CSV_PATH = 'reviews_limpias.csv'

if os.path.exists(CSV_PATH):
    print(f"[Datos] '{CSV_PATH}' encontrado localmente — no se descarga nada.")
else:
    print(f"[Datos] '{CSV_PATH}' no encontrado. Intentando descarga...")
    try:
        import urllib.request, zipfile
        _URL = ("https://drive.google.com/uc?export=download"
                "&id=1RIBhiwKn1cFODUa8QQcV4Y6xPZfgXW7h&confirm=t")
        urllib.request.urlretrieve(_URL, 'reviews_limpias.zip')
        with zipfile.ZipFile('reviews_limpias.zip') as _z:
            _z.extractall('.')
        print("[Datos] Descarga y descompresión completadas.")
    except Exception as _e:
        raise FileNotFoundError(
            f"No se pudo obtener '{CSV_PATH}' (descarga fallida: {_e}). "
            f"Colocá el archivo en el directorio de trabajo y reejecutá."
        )

# ─── Carga ────────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

N_ORIGINAL = len(df)
print("═" * 60)
print(f"  Registros totales : {N_ORIGINAL:,}")
print(f"  Columnas          : {list(df.columns)}")
print("═" * 60)

# ─── Auditoría de calidad ─────────────────────────────────────────────────────
print("\n[Nulos por columna]")
print(df.isnull().sum())

# ═══════════════════════════════════════════════════════════════════════════════
# Reconciliación EXPLÍCITA del N — máscaras calculadas ANTES de borrar
# ═══════════════════════════════════════════════════════════════════════════════
mask_nulos = df['clean_reviews'].isna() | df['sentiment'].isna()
mask_dups  = df.duplicated(subset='clean_reviews')      # marca la 2ª aparición en adelante

N_NULOS      = int(mask_nulos.sum())
N_DUPS       = int(mask_dups.sum())
N_SOLAPAMIENTO = int((mask_nulos & mask_dups).sum())    # filas que son nulas Y duplicadas
N_ELIMINADAS = int((mask_nulos | mask_dups).sum())      # unión, sin doble conteo

print("\n" + "═" * 60)
print("  RECONCILIACIÓN DEL TAMAÑO DEL CORPUS")
print("═" * 60)
print(f"  (A) Registros originales                        : {N_ORIGINAL:,}")
print(f"  (B) Filas con nulos en clean_reviews/sentiment  : {N_NULOS:,}")
print(f"  (C) Filas marcadas como duplicadas              : {N_DUPS:,}")
print(f"  (D) SOLAPAMIENTO  (nulas Y duplicadas a la vez) : {N_SOLAPAMIENTO:,}")
print(f"  (E) Filas eliminadas = unión B ∪ C = B + C − D  : {N_ELIMINADAS:,}")
print("-" * 60)
print(f"  Resta ingenua   A − B − C     = {N_ORIGINAL - N_NULOS - N_DUPS:,}   (doble conteo)")
print(f"  Resta correcta  A − (B+C−D)   = {N_ORIGINAL - N_ELIMINADAS:,}   (N real)")
print("-" * 60)
print("  Por qué difieren: pandas.duplicated() trata NaN == NaN, de modo que todas")
print("  las filas nulas salvo la primera quedan TAMBIÉN marcadas como duplicadas.")
print(f"  Esas {N_SOLAPAMIENTO:,} filas se restarían DOS VECES con la resta ingenua.")
print("═" * 60)

# ─── Eliminación efectiva (unión de ambos criterios) ──────────────────────────
df = df[~(mask_nulos | mask_dups)].reset_index(drop=True)

N_FINAL = len(df)
assert N_FINAL == N_ORIGINAL - N_ELIMINADAS, "La reconciliación no cierra"
print(f"\n✓ Dataset final limpio: {N_FINAL:,} registros")
print(f"✓ Verificación: {N_ORIGINAL:,} − {N_NULOS:,} − {N_DUPS:,} + {N_SOLAPAMIENTO:,} = {N_FINAL:,}")
df.head()


### 1.3 Análisis del Desbalance de Clases y Longitud de Reseñas

In [ ]:
# ─── Estandarizar la columna 'sentiment' a Title Case
df['sentiment'] = df['sentiment'].str.title()

# ─── Mapeo de etiquetas a enteros ─────────────────────────────────────────────
LABEL_MAP = {'Negativo': 0, 'Neutral': 1, 'Positivo': 2}
INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
df['label'] = df['sentiment'].map(LABEL_MAP)

# ─── Longitud en tokens (palabras) ───────────────────────────────────────────
df['n_words'] = df['clean_reviews'].apply(lambda x: len(str(x).split()))

# ─── Figura de EDA ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 5))
gs  = GridSpec(1, 3, figure=fig, wspace=0.35)

COLORS = ['#E74C3C', '#F39C12', '#27AE60']
CLASS_NAMES = ['Negativo', 'Neutral', 'Positivo']

# ── Panel 1: Distribución de clases (absoluta) ─────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
counts = df['sentiment'].value_counts().reindex(CLASS_NAMES)
bars = ax1.bar(CLASS_NAMES, counts.values, color=COLORS, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Distribución de Clases', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cantidad de Reseñas')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Panel 2: Distribución de clases (proporcional) ─────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
pcts = (counts / counts.sum() * 100).round(1)
wedges, texts, autotexts = ax2.pie(
    pcts, labels=CLASS_NAMES, colors=COLORS,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold')
ax2.set_title('Proporción de Clases (%)', fontsize=12, fontweight='bold')

# ── Panel 3: Distribución de longitud por clase ────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
for lbl, color in zip(CLASS_NAMES, COLORS):
    subset = df[df['sentiment'] == lbl]['n_words']
    ax3.hist(subset, bins=50, alpha=0.6, color=color, label=lbl, density=True)

p95 = int(np.percentile(df['n_words'], 95))
ax3.axvline(p95, color='black', linestyle='--', linewidth=1.5, label=f'P95 = {p95} palabras')
ax3.set_title('Longitud de Reseñas por Clase', fontsize=12, fontweight='bold')
ax3.set_xlabel('Número de Palabras')
ax3.set_ylabel('Densidad')
ax3.legend(fontsize=9)
ax3.set_xlim(0, 150)

fig.suptitle('EDA — Corpus de Reseñas Google Play', fontsize=14, fontweight='bold', y=1.02)
Path('figs').mkdir(exist_ok=True)
plt.savefig('figs/exp3_eda_overview.png', dpi=200, bbox_inches='tight')
plt.show()

# ─── Estadísticas descriptivas ───────────────────────────────────────────────
print("\n[Estadísticas de longitud por clase]")
print(df.groupby('sentiment')['n_words'].describe().round(1).to_string())
print(f"\n→ MAX_LEN recomendado (P95): {p95} tokens")

---
## Sección 2 — Preprocesamiento de Texto

### 2.1 Estrategia de Limpieza Textual

In [ ]:
# ─── Hiperparámetros globales de preprocesamiento ────────────────────────────
VOCAB_SIZE  = 10_000   # Tamaño del vocabulario (tokens más frecuentes)
OOV_TOKEN   = '<OOV>'  # Token para palabras fuera del vocabulario
MAX_LEN     = None     # Se asignará al P95 calculado en el EDA

# ─── Función de limpieza ─────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    Limpieza conservadora:
    - Elimina URLs, menciones, hashtags
    - Elimina caracteres no ASCII y números aislados
    - Colapsa espacios múltiples
    - Preserva stopwords para mantener negaciones
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)          # URLs
    text = re.sub(r'@\w+|#\w+', '', text)                  # Menciones y hashtags
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)          # Solo letras y espacios
    text = re.sub(r'\b\w{1}\b', '', text)                  # Tokens de 1 carácter
    text = re.sub(r'\s+', ' ', text).strip()               # Espacios múltiples
    return text

df['text_clean'] = df['clean_reviews'].apply(clean_text)

# ═══════════════════════════════════════════════════════════════════════════════
# Trazabilidad de MAX_LEN: P95 en EDA (texto crudo) vs. P95 aplicado
# ═══════════════════════════════════════════════════════════════════════════════
df['n_words_clean'] = df['text_clean'].apply(lambda x: len(x.split()))

MAX_LEN_EDA      = int(p95)                                        # calculado en celda 6
MAX_LEN_APLICADO = int(np.percentile(df['n_words_clean'], 95))     # tras clean_text()
MAX_LEN          = MAX_LEN_APLICADO                                # el que se usa

print("═" * 68)
print("  TRAZABILIDAD DE MAX_LEN")
print("═" * 68)
print(f"  P95 calculado en el EDA  (sobre 'clean_reviews', texto crudo) : {MAX_LEN_EDA} tokens")
print(f"  P95 aplicado al padding  (sobre 'text_clean', post-limpieza)  : {MAX_LEN_APLICADO} tokens")
print(f"  Diferencia                                                    : {MAX_LEN_EDA - MAX_LEN_APLICADO} tokens")
print("-" * 68)
print("  POR QUÉ DIFIEREN: el P95 del EDA se midió ANTES de clean_text(). La limpieza")
print("  elimina URLs, menciones, hashtags, dígitos, signos de puntuación y todos los")
print("  tokens de 1 carácter, por lo que cada reseña pierde tokens y la distribución")
print("  de longitudes se desplaza a la izquierda. El P95 se RECALCULÓ sobre el texto")
print("  efectivamente tokenizado, que es el único consistente con lo que ve el modelo.")
print(f"  → Se usa MAX_LEN = {MAX_LEN}. Usar {MAX_LEN_EDA} habría añadido "
      f"{MAX_LEN_EDA - MAX_LEN_APLICADO} pasos de padding puro por secuencia.")
print("-" * 68)
_media_crudo   = df['n_words'].mean()
_media_limpio  = df['n_words_clean'].mean()
print(f"  Longitud media  crudo / limpio : {_media_crudo:.2f} → {_media_limpio:.2f} tokens "
      f"({(_media_limpio/_media_crudo - 1)*100:+.1f}%)")
_truncadas = int((df['n_words_clean'] > MAX_LEN).sum())
print(f"  Reseñas truncadas por MAX_LEN  : {_truncadas:,} / {len(df):,} "
      f"({_truncadas/len(df)*100:.2f}%)")
print("═" * 68)

# Muestra de transformación
print("\n[Ejemplo de limpieza]")
idx = df[df['n_words'] > 10].index[0]
print(f"  Original : {df.loc[idx, 'clean_reviews'][:120]}")
print(f"  Limpio   : {df.loc[idx, 'text_clean'][:120]}")

### 2.2 Tokenización con Keras y Representación como Embedding


In [ ]:
# ─── Partición estratificada Train / Val / Test ───────────────────────────────
# Estratificación obligatoria dado el desbalance de clases
X = df['text_clean'].values
y = df['label'].values

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15, random_state=SEED, stratify=y_train_val
)

print("[Partición del Dataset]")
for split, arr in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    total = len(arr)
    cnts  = Counter(arr)
    pcts  = {INV_LABEL[k]: f"{v/total*100:.1f}%" for k, v in sorted(cnts.items())}
    print(f"  {split:<6}: {total:>6,} registros | Distribución: {pcts}")

# ─── Tokenizador ─────────────────────────────────────────────────────────────
# IMPORTANTE: el tokenizador se ajusta SOLO sobre train para evitar data leakage
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train)

vocab_actual = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print(f"\n✓ Tokenizador ajustado.")
print(f"  Vocabulario corpus    : {len(tokenizer.word_index):,} tokens únicos")
print(f"  Vocabulario efectivo  : {vocab_actual:,} tokens (top-{VOCAB_SIZE})")

# ─── Secuencias + Padding ─────────────────────────────────────────────────────
def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_seq = encode(X_train)
X_val_seq   = encode(X_val)
X_test_seq  = encode(X_test)

print(f"\n✓ Padding aplicado. Shape de X_train: {X_train_seq.shape}")

# ─── Class Weights para mitigar el desbalance ─────────────────────────────────
# compute_class_weight calcula pesos inversamente proporcionales a la frecuencia
# de cada clase, penalizando más los errores en la clase minoritaria (Neutral).
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_arr))

print("\n[Pesos de clase calculados]")
for k, v in class_weight_dict.items():
    print(f"  Clase {k} ({INV_LABEL[k]:<10}): peso = {v:.4f}")

---
## Sección 3 — Modelo Baseline: Bag of Words (TF-IDF + Regresión Logística)

In [ ]:
# ─── Vectorización TF-IDF ────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features   = 15_000,
    ngram_range     = (1, 2),   # Unigramas + bigramas para capturar algo de contexto local
    sublinear_tf    = True,     # log(TF) para comprimir magnitudes
    min_df          = 3,        # Ignorar tokens que aparecen menos de 3 veces
    strip_accents   = 'unicode'
)

# Ajuste SOLO sobre train
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"✓ TF-IDF. Shape X_train: {X_train_tfidf.shape}")

# ─── Regresión Logística ─────────────────────────────────────────────────────
lr_model = LogisticRegression(
    C              = 1.0,
    max_iter       = 1000,
    solver         = 'lbfgs',
    class_weight   = 'balanced',   # Equivalente al class_weight_dict para sklearn
    random_state   = SEED,
    n_jobs         = -1
)

lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"\n[Accuracy en Test — BoW + LR] : {acc_lr:.4f} ({acc_lr*100:.2f}%)")

# ─── Reporte de clasificación ─────────────────────────────────────────────────
print("\n[Reporte de Clasificación]")
print(classification_report(
    y_test, y_pred_lr,
    target_names=CLASS_NAMES,
    digits=4
))

# ─── Matriz de confusión ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_lr, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2%')
ax.set_title('Matriz de Confusión Normalizada\nBaseline: TF-IDF + Regresión Logística',
             fontweight='bold')
plt.tight_layout()
plt.savefig('figs/exp3_cm_baseline.png', dpi=200, bbox_inches='tight')
plt.show()

# ─── Top palabras por clase (interpretabilidad) ───────────────────────────────
feature_names = np.array(tfidf.get_feature_names_out())
print("\n[Top-10 términos más discriminativos por clase]")
for i, clase in enumerate(CLASS_NAMES):
    coefs = lr_model.coef_[i]
    top_idx = np.argsort(coefs)[-10:][::-1]
    print(f"  {clase}: {', '.join(feature_names[top_idx])}")

---
## Sección 4 — Modelo RNA Secuencial: Custom Embedding + BiLSTM

### 4.1 Arquitectura y Justificación

#### Descripción de las capas

| Capa | Función |
|---|---|
| `Embedding(VOCAB_SIZE, EMBED_DIM)` | Aprende representaciones vectoriales densas de tokens. Transforma cada índice en un vector de `EMBED_DIM` dimensiones ajustable por backprop. |
| `Dropout(0.3)` | Regularización estocástica: desactiva aleatoriamente el 30% de las activaciones durante el entrenamiento para prevenir el sobreajuste. |
| `Bidirectional(LSTM(units))` | Procesa la secuencia en ambas direcciones capturando dependencias contextuales de largo alcance. |
| `Dense(64, relu)` | Capa de proyección no-lineal que comprime la representación antes de la clasificación. |
| `Dense(3, softmax)` | Capa de salida que produce una distribución de probabilidad sobre las 3 clases. |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Instrumentación temporal — entrenamiento, épocas y latencia
# ═══════════════════════════════════════════════════════════════════════════════
import time, platform, json

# Registro global de mediciones
TIEMPOS = {}      # nombre_modelo -> dict con métricas temporales

# ─── Acelerador detectado ─────────────────────────────────────────────────────
_gpus = tf.config.list_physical_devices('GPU')
ACELERADOR = f"GPU ({_gpus[0].name})" if _gpus else "CPU"
print(f"[Entorno] Acelerador detectado : {ACELERADOR}")
print(f"[Entorno] Python              : {platform.python_version()}")
print(f"[Entorno] TensorFlow          : {tf.__version__}")

# ─── Callback que cronometra cada época ───────────────────────────────────────
class CronometroEpocas(tf.keras.callbacks.Callback):
    """Guarda la duración (s) de cada época para poder reportar s/época."""
    def __init__(self):
        super().__init__()
        self.duraciones = []
    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.duraciones.append(time.perf_counter() - self._t0)

# ─── Latencia de inferencia con calentamiento ─────────────────────────────────
def medir_latencia_keras(model, X, batch_size, n_rep=10, n_warmup=3):
    """
    Latencia de inferencia de un modelo Keras.
    batch_size=64  -> throughput por lote
    batch_size=1   -> latencia unitaria (peor caso online)
    Devuelve (ms_por_lote_mediana, ms_por_muestra).
    """
    if batch_size == 1:
        muestras = X[:100]                       # 100 predicciones unitarias
        for _ in range(n_warmup):                # calentamiento (grafo + XLA + cuDNN)
            model.predict(muestras[:1], verbose=0)
        tiempos = []
        for i in range(len(muestras)):
            t0 = time.perf_counter()
            model.predict(muestras[i:i+1], verbose=0)
            tiempos.append((time.perf_counter() - t0) * 1000)
        ms_lote = float(np.median(tiempos))
        return ms_lote, ms_lote
    lote = X[:batch_size]
    for _ in range(n_warmup):
        model.predict(lote, batch_size=batch_size, verbose=0)
    tiempos = []
    for _ in range(n_rep):
        t0 = time.perf_counter()
        model.predict(lote, batch_size=batch_size, verbose=0)
        tiempos.append((time.perf_counter() - t0) * 1000)
    ms_lote = float(np.median(tiempos))
    return ms_lote, ms_lote / batch_size

def medir_latencia_sklearn(vectorizer, model, textos, batch_size, n_rep=10, n_warmup=3):
    """
    Latencia del pipeline TF-IDF + LR. IMPORTANTE: incluye la vectorización,
    porque en producción el texto crudo también hay que transformarlo.
    """
    n = batch_size if batch_size > 1 else 1
    lote = textos[:n]
    for _ in range(n_warmup):
        model.predict(vectorizer.transform(lote))
    tiempos = []
    reps = n_rep if batch_size > 1 else 100
    for i in range(reps):
        sub = textos[i:i+1] if batch_size == 1 else lote
        t0 = time.perf_counter()
        model.predict(vectorizer.transform(sub))
        tiempos.append((time.perf_counter() - t0) * 1000)
    ms_lote = float(np.median(tiempos))
    return ms_lote, ms_lote / n

def resumir_entrenamiento(nombre, history, cronometro, segundos_totales):
    """Consolida tiempo de entrenamiento y épocas hasta la mejor (min val_loss)."""
    val_loss   = history.history['val_loss']
    epocas_tot = len(val_loss)
    mejor_ep   = int(np.argmin(val_loss)) + 1        # 1-indexado
    TIEMPOS[nombre] = {
        'segundos_entrenamiento' : round(segundos_totales, 4),
        'epocas_ejecutadas'      : epocas_tot,
        'epoca_mejor_val_loss'   : mejor_ep,
        'segundos_por_epoca'     : round(float(np.mean(cronometro.duraciones)), 4),
        'segundos_hasta_mejor'   : round(float(np.sum(cronometro.duraciones[:mejor_ep])), 4),
        'mejor_val_loss'         : round(float(np.min(val_loss)), 4),
    }
    print(f"\n[Tiempos — {nombre}]")
    for k, v in TIEMPOS[nombre].items():
        print(f"  {k:<24}: {v}")
    return TIEMPOS[nombre]

print("✓ Utilidades de medición temporal listas.")


In [ ]:
# ─── Hiperparámetros del modelo ───────────────────────────────────────────────
EMBED_DIM    = 128
LSTM_UNITS   = 64
DROPOUT_RATE = 0.3
NUM_CLASSES  = 3
BATCH_SIZE   = 128
EPOCHS       = 30

# ─── Construcción de la arquitectura BiLSTM ───────────────────────────────────
def build_bilstm(vocab_size, embed_dim, max_len, lstm_units,
                 dropout_rate, num_classes):
    inputs = Input(shape=(max_len,), name='input_tokens')

    # Capa Embedding: convierte índices en vectores densos aprendibles
    x = Embedding(
        input_dim    = vocab_size,
        output_dim   = embed_dim,
        input_length = max_len,
        name         = 'embedding'
    )(inputs)
    x = Dropout(dropout_rate, name='dropout_embed')(x)

    # BiLSTM: procesa la secuencia en ambas direcciones
    x = Bidirectional(
        LSTM(lstm_units, return_sequences=False),
        name='bilstm'
    )(x)
    x = Dropout(dropout_rate, name='dropout_lstm')(x)

    # Capas densas de clasificación
    x = Dense(64, activation='relu', name='dense_proj')(x)
    x = BatchNormalization(name='batch_norm')(x)
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='BiLSTM_Sentiment')
    return model

bilstm_model = build_bilstm(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN,
    lstm_units=LSTM_UNITS, dropout_rate=DROPOUT_RATE, num_classes=NUM_CLASSES
)
bilstm_model.summary()

# ─── Compilación ─────────────────────────────────────────────────────────────
bilstm_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

# ─── Callbacks ────────────────────────────────────────────────────────────────
callbacks_bilstm = [
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.5,
        patience  = 3,
        min_lr    = 1e-6,
        verbose   = 1
    ),
    ModelCheckpoint(
        filepath             = 'best_bilstm.keras',
        monitor              = 'val_accuracy',
        save_best_only       = True,
        verbose              = 0
    ),
    CronometroEpocas()                      #
]
crono_bilstm = callbacks_bilstm[-1]         # referencia al cronómetro

# ─── Entrenamiento ────────────────────────────────────────────────────────────
print("\n[Iniciando entrenamiento BiLSTM...]\n")
_t0_bilstm = time.perf_counter()                                    #
history_bilstm = bilstm_model.fit(
    X_train_seq, y_train,
    validation_data = (X_val_seq, y_val),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weight_dict,
    callbacks       = callbacks_bilstm,
    verbose         = 1
)
t_train_bilstm = time.perf_counter() - _t0_bilstm                   #
resumir_entrenamiento('BiLSTM', history_bilstm, crono_bilstm, t_train_bilstm)


In [ ]:
# ─── Curvas de aprendizaje ────────────────────────────────────────────────────
def plot_history(history, model_name, color='steelblue'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, title in zip(
        axes,
        [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
        ['Función de Pérdida', 'Accuracy']
    ):
        train_key, val_key = metric
        ax.plot(history.history[train_key],   label='Train', color=color, lw=2)
        ax.plot(history.history[val_key], label='Validación',
                color=color, lw=2, linestyle='--', alpha=0.7)
        best_epoch = np.argmin(history.history['val_loss'])
        ax.axvline(best_epoch, color='red', linestyle=':', alpha=0.6, label=f'Mejor época ({best_epoch+1})')
        ax.set_title(f'{model_name} — {title}', fontweight='bold')
        ax.set_xlabel('Época')
        ax.legend()
    plt.tight_layout()
    plt.savefig(f'figs/exp3_history_{model_name.lower().replace(" ","_")}.png', dpi=200, bbox_inches='tight')
    plt.show()

plot_history(history_bilstm, 'BiLSTM')

# ─── Evaluación en Test ───────────────────────────────────────────────────────
y_pred_bilstm_probs = bilstm_model.predict(X_test_seq, batch_size=256, verbose=0)
y_pred_bilstm       = np.argmax(y_pred_bilstm_probs, axis=1)
acc_bilstm          = accuracy_score(y_test, y_pred_bilstm)

print(f"[Accuracy en Test — BiLSTM] : {acc_bilstm:.4f} ({acc_bilstm*100:.2f}%)")
print("\n[Reporte de Clasificación — BiLSTM]")
print(classification_report(y_test, y_pred_bilstm, target_names=CLASS_NAMES, digits=4))

---
## Sección 5 — Modelo RNA Patrones Locales: Custom Embedding + CNN-1D


In [ ]:
# ─── Hiperparámetros CNN ──────────────────────────────────────────────────────
NUM_FILTERS  = 128
KERNEL_SIZE  = 3   # Detecta trigramas

# ─── Construcción de la arquitectura CNN-1D ───────────────────────────────────
def build_cnn1d(vocab_size, embed_dim, max_len, num_filters,
                kernel_size, dropout_rate, num_classes):
    inputs = Input(shape=(max_len,), name='input_tokens')

    # Embedding compartido con la misma filosofía que en BiLSTM
    x = Embedding(
        input_dim    = vocab_size,
        output_dim   = embed_dim,
        input_length = max_len,
        name         = 'embedding'
    )(inputs)
    x = Dropout(dropout_rate, name='dropout_embed')(x)

    # Conv1D: cada filtro aprende un patrón de kernel_size tokens consecutivos
    x = Conv1D(
        filters     = num_filters,
        kernel_size = kernel_size,
        activation  = 'relu',
        padding     = 'same',
        name        = 'conv1d_trigram'
    )(x)

    # GlobalMaxPooling: extrae la activación máxima de cada filtro (invariancia posicional)
    x = GlobalMaxPooling1D(name='global_max_pool')(x)
    x = Dropout(dropout_rate, name='dropout_pool')(x)

    # Capas densas de clasificación
    x = Dense(64, activation='relu', name='dense_proj')(x)
    x = BatchNormalization(name='batch_norm')(x)
    outputs = Dense(num_classes, activation='softmax', name='output')(x)

    model = Model(inputs=inputs, outputs=outputs, name='CNN1D_Sentiment')
    return model

cnn_model = build_cnn1d(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=MAX_LEN,
    num_filters=NUM_FILTERS, kernel_size=KERNEL_SIZE,
    dropout_rate=DROPOUT_RATE, num_classes=NUM_CLASSES
)
cnn_model.summary()

# ─── Compilación ─────────────────────────────────────────────────────────────
cnn_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

# ─── Callbacks ────────────────────────────────────────────────────────────────
callbacks_cnn = [
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = 5,
        restore_best_weights = True,
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.5,
        patience  = 3,
        min_lr    = 1e-6,
        verbose   = 1
    ),
    ModelCheckpoint(
        filepath       = 'best_cnn1d.keras',
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 0
    ),
    CronometroEpocas()                      #
]
crono_cnn = callbacks_cnn[-1]               #

# ─── Entrenamiento ────────────────────────────────────────────────────────────
print("\n[Iniciando entrenamiento CNN-1D...]\n")
_t0_cnn = time.perf_counter()                                       #
history_cnn = cnn_model.fit(
    X_train_seq, y_train,
    validation_data = (X_val_seq, y_val),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    class_weight    = class_weight_dict,
    callbacks       = callbacks_cnn,
    verbose         = 1
)
t_train_cnn = time.perf_counter() - _t0_cnn                         #
resumir_entrenamiento('CNN-1D', history_cnn, crono_cnn, t_train_cnn)

plot_history(history_cnn, 'CNN-1D', color='darkorange')

# ─── Evaluación en Test ───────────────────────────────────────────────────────
y_pred_cnn_probs = cnn_model.predict(X_test_seq, batch_size=256, verbose=0)
y_pred_cnn       = np.argmax(y_pred_cnn_probs, axis=1)
acc_cnn          = accuracy_score(y_test, y_pred_cnn)

print(f"[Accuracy en Test — CNN-1D] : {acc_cnn:.4f} ({acc_cnn*100:.2f}%)")
print("\n[Reporte de Clasificación — CNN-1D]")
print(classification_report(y_test, y_pred_cnn, target_names=CLASS_NAMES, digits=4))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Tiempo de entrenamiento del baseline + latencias de inferencia
# ═══════════════════════════════════════════════════════════════════════════════
# ─── (1) Re-entrenar el baseline SOLO para cronometrarlo ──────────────────────
# Se usa una copia con los mismos hiperparámetros: no toca lr_model ni y_pred_lr,
# por lo que las métricas ya reportadas NO cambian.
from sklearn.base import clone
_lr_crono = clone(lr_model)
_t0 = time.perf_counter()
_lr_crono.fit(X_train_tfidf, y_train)
t_fit_lr = time.perf_counter() - _t0

_t0 = time.perf_counter()
_ = tfidf.fit_transform(X_train)          # costo de vectorizar el corpus de train
t_vec_lr = time.perf_counter() - _t0

TIEMPOS['TF-IDF + Reg. Logística'] = {
    'segundos_entrenamiento' : round(t_fit_lr + t_vec_lr, 4),
    'segundos_vectorizacion' : round(t_vec_lr, 4),
    'segundos_ajuste_lr'     : round(t_fit_lr, 4),
    'epocas_ejecutadas'      : int(_lr_crono.n_iter_.max()),   # iters lbfgs, NO épocas
    'epoca_mejor_val_loss'   : None,                            # no aplica a lbfgs
    'segundos_por_epoca'     : None,
    'segundos_hasta_mejor'   : round(t_fit_lr + t_vec_lr, 4),
    'mejor_val_loss'         : None,
}
print(f"[TF-IDF + RL] entrenamiento total: {t_fit_lr + t_vec_lr:.4f} s "
      f"(vectorización {t_vec_lr:.4f} s + ajuste {t_fit_lr:.4f} s), "
      f"iteraciones lbfgs: {_lr_crono.n_iter_.max()}")

# ─── (2) Latencias de inferencia ──────────────────────────────────────────────
LATENCIAS = {}

for _nombre, _modelo in [('BiLSTM', bilstm_model), ('CNN-1D', cnn_model)]:
    ms64, ms64_u = medir_latencia_keras(_modelo, X_test_seq, batch_size=64)
    ms1,  _      = medir_latencia_keras(_modelo, X_test_seq, batch_size=1)
    LATENCIAS[_nombre] = {
        'ms_lote64'          : round(ms64, 4),
        'ms_por_muestra_b64' : round(ms64_u, 4),
        'ms_batch1'          : round(ms1, 4),
    }

ms64, ms64_u = medir_latencia_sklearn(tfidf, lr_model, X_test, batch_size=64)
ms1,  _      = medir_latencia_sklearn(tfidf, lr_model, X_test, batch_size=1)
LATENCIAS['TF-IDF + Reg. Logística'] = {
    'ms_lote64'          : round(ms64, 4),
    'ms_por_muestra_b64' : round(ms64_u, 4),
    'ms_batch1'          : round(ms1, 4),
}

print("\n[Latencia de inferencia — mediana, con calentamiento]")
print(f"{'Modelo':<26}{'ms/lote(64)':>13}{'ms/muestra(64)':>16}{'ms/muestra(b=1)':>17}")
for k, v in LATENCIAS.items():
    print(f"{k:<26}{v['ms_lote64']:>13.3f}{v['ms_por_muestra_b64']:>16.4f}{v['ms_batch1']:>17.3f}")
print("\nNota: la latencia de TF-IDF+RL INCLUYE la vectorización del texto crudo "
      "(en producción es obligatoria). Las de BiLSTM/CNN parten de secuencias ya "
      "tokenizadas y padeadas, por lo que están sesgadas a favor de las redes.")


---
## Sección 6 — Evaluación Comparativa y Análisis de Errores


In [ ]:
# Complejidad de cada modelo (dimensión "costo" de la tabla)
PARAMS_MODELO = {
    'BiLSTM' : int(sum(np.prod(w.shape) for w in bilstm_model.trainable_weights)),
    'CNN-1D' : int(sum(np.prod(w.shape) for w in cnn_model.trainable_weights)),
    'TF-IDF + Reg. Logística' : int(lr_model.coef_.size + lr_model.intercept_.size),
}
for k, v in PARAMS_MODELO.items():
    print(f"  {k:<26}: {v:,} parámetros entrenables")


In [ ]:
from sklearn.metrics import f1_score

# ─── Tabla comparativa ────────────────────────────────────────────────────────
results = {
    'TF-IDF + Reg. Logística': {
        'y_pred': y_pred_lr,
        'color': '#3498DB'
    },
    'BiLSTM': {
        'y_pred': y_pred_bilstm,
        'color': '#E74C3C'
    },
    'CNN-1D': {
        'y_pred': y_pred_cnn,
        'color': '#F39C12'
    }
}

# ═══════════════════════════════════════════════════════════════════════════════
# Tabla comparativa generada desde classification_report (métricas REALES)
# ═══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import precision_recall_fscore_support

rows = []
REPORTES = {}     # nombre -> dict del classification_report

for name, data in results.items():
    y_p = data['y_pred']
    # F1 clase Neutral tomado LITERALMENTE del classification_report, no recalculado aparte
    rep = classification_report(y_test, y_p, target_names=CLASS_NAMES,
                                digits=4, output_dict=True, zero_division=0)
    REPORTES[name] = rep
    t = TIEMPOS.get(name, {})
    l = LATENCIAS.get(name, {})
    rows.append({
        'Modelo'              : name,
        'Accuracy'            : round(rep['accuracy'], 4),
        'F1-Macro'            : round(rep['macro avg']['f1-score'], 4),
        'F1-Negativo'         : round(rep['Negativo']['f1-score'], 4),
        'F1-Neutral'          : round(rep['Neutral']['f1-score'], 4),   #
        'F1-Positivo'         : round(rep['Positivo']['f1-score'], 4),
        'Recall-Neutral'      : round(rep['Neutral']['recall'], 4),
        'Precision-Neutral'   : round(rep['Neutral']['precision'], 4),
        'Entrenamiento (s)'   : t.get('segundos_entrenamiento'),
        'Época mejor val_loss': t.get('epoca_mejor_val_loss'),
        'Inferencia ms/lote64': l.get('ms_lote64'),
        'Inferencia ms/b=1'   : l.get('ms_batch1'),
        'Params entrenables'  : PARAMS_MODELO.get(name),
    })

df_results = pd.DataFrame(rows).set_index('Modelo')

print("[Tabla Comparativa Multidimensional — Conjunto de Test] (métricas medidas, no cualitativas)")
print(df_results.to_string())

# ─── Ranking explícito por la dimensión que decide la recomendación ──────────
print("\n[Ranking por F1 clase Neutral — orden REAL]")
for pos, (m, v) in enumerate(df_results['F1-Neutral'].sort_values(ascending=False).items(), 1):
    print(f"  {pos}. {m:<26} F1-Neutral = {v:.4f}")
print("\n[Ranking por Macro-F1 — orden REAL]")
for pos, (m, v) in enumerate(df_results['F1-Macro'].sort_values(ascending=False).items(), 1):
    print(f"  {pos}. {m:<26} Macro-F1 = {v:.4f}")

_mejor_macro = df_results['F1-Macro'].idxmax()
_mejor_acc   = df_results['Accuracy'].idxmax()
print(f"\n→ Mejor Macro-F1 : {_mejor_macro} ({df_results.loc[_mejor_macro,'F1-Macro']:.4f})")
print(f"→ Mejor Accuracy : {_mejor_acc} ({df_results.loc[_mejor_acc,'Accuracy']:.4f})")
if _mejor_macro != _mejor_acc:
    print("⚠ Accuracy y Macro-F1 señalan modelos DISTINTOS: con 6.7% de Neutral en test, "
          "accuracy premia ignorar la clase minoritaria. La recomendación debe basarse en Macro-F1.")
_colapsan = df_results.index[df_results['Recall-Neutral'] == 0.0].tolist()
if _colapsan:
    print(f"⚠ COLAPSO TOTAL de la clase Neutral (recall = 0) en: {', '.join(_colapsan)}. "
          f"Ese modelo NO puede recomendarse para producción sin mitigación.")

# ─── Visualización comparativa ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Accuracy y F1-Macro por modelo
x = np.arange(len(results))
width = 0.35
colors_list = [d['color'] for d in results.values()]

bars1 = axes[0].bar(x - width/2, df_results['Accuracy'], width,
                    label='Accuracy', color=colors_list, alpha=0.85)
bars2 = axes[0].bar(x + width/2, df_results['F1-Macro'], width,
                    label='F1-Macro', color=colors_list, alpha=0.5, hatch='//')
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.003,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=8.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(list(results.keys()), rotation=10, ha='right', fontsize=9)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Accuracy y F1-Macro por Modelo', fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Score')

# Panel 2: F1 por clase (foco en Neutral)
metrics_cls = ['F1-Negativo', 'F1-Neutral', 'F1-Positivo']
x2    = np.arange(len(metrics_cls))
width2 = 0.25
for i, (name, data) in enumerate(results.items()):
    vals = [df_results.loc[name, m] for m in metrics_cls]
    axes[1].bar(x2 + i * width2 - width2, vals, width2,
                label=name, color=data['color'], alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(['Negativo', 'Neutral', 'Positivo'])
axes[1].set_ylim(0, 1.1)
axes[1].set_title('F1-Score por Clase', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_ylabel('F1-Score')
axes[1].axvline(0.5, color='gray', linestyle=':', alpha=0.5)  # Separador visual

fig.suptitle('Comparativa de Modelos — Conjunto de Test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figs/exp3_model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Umbralización de la clase Neutral
#   Regla: si P(Neutral) >= t  →  predecir Neutral
#          si no               →  argmax entre {Negativo, Positivo}
#   El umbral t se elige en VALIDACIÓN (maximiza macro-F1) y se aplica en TEST.
# ═══════════════════════════════════════════════════════════════════════════════
IDX_NEUTRAL = LABEL_MAP['Neutral']      # = 1
IDX_NO_NEUTRAL = [LABEL_MAP['Negativo'], LABEL_MAP['Positivo']]   # [0, 2]

def predecir_con_umbral(probs, t):
    """Aplica la regla de umbral sobre la probabilidad softmax de Neutral."""
    probs = np.asarray(probs)
    # argmax restringido a las clases NO-Neutral
    sub = probs[:, IDX_NO_NEUTRAL]
    pred = np.array(IDX_NO_NEUTRAL)[np.argmax(sub, axis=1)]
    pred = np.where(probs[:, IDX_NEUTRAL] >= t, IDX_NEUTRAL, pred)
    return pred

def barrer_umbral(probs_val, y_val_, grid=None):
    """Barre t en validación y devuelve (t_optimo, DataFrame del barrido)."""
    if grid is None:
        grid = np.round(np.arange(0.02, 0.905, 0.01), 3)
    filas = []
    for t in grid:
        p = predecir_con_umbral(probs_val, t)
        rep = classification_report(y_val_, p, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
        filas.append({
            'umbral'        : float(t),
            'macro_f1'      : rep['macro avg']['f1-score'],
            'f1_neutral'    : rep['Neutral']['f1-score'],
            'recall_neutral': rep['Neutral']['recall'],
            'precision_neutral': rep['Neutral']['precision'],
            'accuracy'      : rep['accuracy'],
        })
    df_barrido = pd.DataFrame(filas)
    t_opt = float(df_barrido.loc[df_barrido['macro_f1'].idxmax(), 'umbral'])
    return t_opt, df_barrido

def evaluar_umbralizacion(nombre, model, X_val_s, y_val_, X_test_s, y_test_):
    """Pipeline completo: probs val → barrido → t* → reevaluación en TEST."""
    probs_val  = model.predict(X_val_s,  batch_size=256, verbose=0)
    probs_test = model.predict(X_test_s, batch_size=256, verbose=0)

    t_opt, df_barrido = barrer_umbral(probs_val, y_val_)

    # ANTES: argmax puro
    pred_antes = np.argmax(probs_test, axis=1)
    rep_antes  = classification_report(y_test_, pred_antes, target_names=CLASS_NAMES,
                                       output_dict=True, zero_division=0)
    # DESPUÉS: umbral óptimo elegido en VALIDACIÓN
    pred_desp  = predecir_con_umbral(probs_test, t_opt)
    rep_desp   = classification_report(y_test_, pred_desp, target_names=CLASS_NAMES,
                                       output_dict=True, zero_division=0)

    print("═" * 78)
    print(f"[UMBRALIZACIÓN NEUTRAL — {nombre}]")
    print(f"  Umbral óptimo (elegido en VALIDACIÓN, {len(y_val_)} muestras): t* = {t_opt:.2f}")
    print(f"  Macro-F1 en validación con t*: {df_barrido['macro_f1'].max():.4f} "
          f"(con argmax/t=0.50: {df_barrido.loc[df_barrido['umbral']==0.50,'macro_f1'].values[0]:.4f})")
    print(f"\n  {'Métrica en TEST':<24}{'argmax (antes)':>16}{'umbral t* (después)':>22}{'Δ':>10}")
    for etiqueta, clave in [('Accuracy', None), ('Macro-F1', 'macro'),
                            ('F1 Neutral', 'f1n'), ('Recall Neutral', 'rn'),
                            ('Precision Neutral', 'pn')]:
        if clave is None:
            a, d = rep_antes['accuracy'], rep_desp['accuracy']
        elif clave == 'macro':
            a, d = rep_antes['macro avg']['f1-score'], rep_desp['macro avg']['f1-score']
        elif clave == 'f1n':
            a, d = rep_antes['Neutral']['f1-score'], rep_desp['Neutral']['f1-score']
        elif clave == 'rn':
            a, d = rep_antes['Neutral']['recall'], rep_desp['Neutral']['recall']
        else:
            a, d = rep_antes['Neutral']['precision'], rep_desp['Neutral']['precision']
        print(f"  {etiqueta:<24}{a:>16.4f}{d:>22.4f}{d-a:>+10.4f}")

    print(f"\n  [Reporte completo en TEST con t* = {t_opt:.2f}]")
    print(classification_report(y_test_, pred_desp, target_names=CLASS_NAMES,
                                digits=4, zero_division=0))
    return {
        'nombre': nombre, 'umbral_optimo': t_opt, 'barrido': df_barrido,
        'pred_antes': pred_antes, 'pred_despues': pred_desp,
        'rep_antes': rep_antes, 'rep_despues': rep_desp,
        'probs_val': probs_val, 'probs_test': probs_test,
    }

UMBRAL = {}
UMBRAL['BiLSTM'] = evaluar_umbralizacion('BiLSTM', bilstm_model,
                                         X_val_seq, y_val, X_test_seq, y_test)
UMBRAL['CNN-1D'] = evaluar_umbralizacion('CNN-1D', cnn_model,
                                         X_val_seq, y_val, X_test_seq, y_test)

# ─── Curva del barrido ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (nombre, r) in zip(axes, UMBRAL.items()):
    b = r['barrido']
    ax.plot(b['umbral'], b['macro_f1'],   label='Macro-F1 (val)', lw=2, color='#2C3E50')
    ax.plot(b['umbral'], b['f1_neutral'], label='F1 Neutral (val)', lw=2,
            color='#F39C12', linestyle='--')
    ax.plot(b['umbral'], b['accuracy'],   label='Accuracy (val)', lw=1.5,
            color='#95A5A6', alpha=0.7)
    ax.axvline(r['umbral_optimo'], color='red', linestyle=':', lw=2,
               label=f"t* = {r['umbral_optimo']:.2f}")
    ax.axvline(0.5, color='gray', linestyle='-.', lw=1, alpha=0.6, label='argmax ≈ 0.50')
    ax.set_title(f'{nombre} — barrido de umbral Neutral', fontweight='bold')
    ax.set_xlabel('Umbral sobre P(Neutral)')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Score en VALIDACIÓN')
fig.suptitle('Selección del umbral de la clase Neutral (validación)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figs/exp3_barrido_umbral_neutral.png', dpi=200, bbox_inches='tight')   # [Z.3]
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Ablación de estrategias contra el desbalance
#   E1: sin mitigación   (argmax, SIN class_weight)
#   E2: class_weight     (argmax, CON class_weight)
#   E3: class_weight + umbralización de Neutral
# ═══════════════════════════════════════════════════════════════════════════════
def entrenar_sin_pesos(constructor, nombre, **kwargs_arq):
    """Reentrena la MISMA arquitectura sin class_weight, misma semilla y callbacks."""
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    modelo = constructor(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM,
                         max_len=MAX_LEN, dropout_rate=DROPOUT_RATE,
                         num_classes=NUM_CLASSES, **kwargs_arq)
    modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                   loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    crono = CronometroEpocas()
    cbs = [EarlyStopping(monitor='val_loss', patience=5,
                         restore_best_weights=True, verbose=0),
           ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                             min_lr=1e-6, verbose=0),
           crono]
    print(f"\n[Entrenando {nombre} SIN class_weight...]")
    t0 = time.perf_counter()
    hist = modelo.fit(X_train_seq, y_train,
                      validation_data=(X_val_seq, y_val),
                      epochs=EPOCHS, batch_size=BATCH_SIZE,
                      callbacks=cbs, verbose=0)          # SIN class_weight
    seg = time.perf_counter() - t0
    print(f"  ✓ {len(hist.history['loss'])} épocas en {seg:.2f} s")
    return modelo, hist, seg

bilstm_sin_pesos, hist_bilstm_sp, t_bilstm_sp = entrenar_sin_pesos(
    build_bilstm, 'BiLSTM', lstm_units=LSTM_UNITS)
cnn_sin_pesos, hist_cnn_sp, t_cnn_sp = entrenar_sin_pesos(
    build_cnn1d, 'CNN-1D', num_filters=NUM_FILTERS, kernel_size=KERNEL_SIZE)

# ─── Consolidar las tres estrategias ──────────────────────────────────────────
ABLACION = []

def _fila(modelo_nombre, estrategia, y_pred_):
    rep = classification_report(y_test, y_pred_, target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)
    return {
        'Modelo'         : modelo_nombre,
        'Estrategia'     : estrategia,
        'Accuracy'       : round(rep['accuracy'], 4),
        'Macro-F1'       : round(rep['macro avg']['f1-score'], 4),
        'F1-Neutral'     : round(rep['Neutral']['f1-score'], 4),
        'Recall-Neutral' : round(rep['Neutral']['recall'], 4),
        'Precision-Neutral': round(rep['Neutral']['precision'], 4),
        'Pred. Neutral (n)': int((y_pred_ == IDX_NEUTRAL).sum()),
    }

for nom, mod_sp, mod_cw in [('BiLSTM', bilstm_sin_pesos, bilstm_model),
                            ('CNN-1D', cnn_sin_pesos, cnn_model)]:
    p_sp = np.argmax(mod_sp.predict(X_test_seq, batch_size=256, verbose=0), axis=1)
    ABLACION.append(_fila(nom, 'E1 · sin mitigación', p_sp))
    ABLACION.append(_fila(nom, 'E2 · class_weight', UMBRAL[nom]['pred_antes']))
    ABLACION.append(_fila(nom, f"E3 · class_weight + umbral t*={UMBRAL[nom]['umbral_optimo']:.2f}",
                          UMBRAL[nom]['pred_despues']))

# Referencia: el baseline, que ya usa class_weight='balanced' en sklearn
ABLACION.append(_fila('TF-IDF + Reg. Logística', 'E2 · class_weight', y_pred_lr))

df_ablacion = pd.DataFrame(ABLACION)
print("\n" + "═" * 96)
print("  ABLACIÓN DE ESTRATEGIAS CONTRA EL DESBALANCE — Conjunto de Test "
      f"(soporte Neutral = {int((y_test == IDX_NEUTRAL).sum())} muestras)")
print("═" * 96)
print(df_ablacion.to_string(index=False))
print("═" * 96)
print("Lectura: 'Pred. Neutral (n)' = cuántas muestras de test recibieron la etiqueta")
print("Neutral. Si vale 0, el modelo colapsó la clase minoritaria por completo.")

# ─── Gráfico de la ablación ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metrica in zip(axes, ['Macro-F1', 'F1-Neutral']):
    sub = df_ablacion[df_ablacion['Modelo'].isin(['BiLSTM', 'CNN-1D'])]
    pivot = sub.pivot_table(index='Modelo', columns='Estrategia',
                            values=metrica, aggfunc='first')
    pivot.plot(kind='bar', ax=ax, rot=0, edgecolor='white',
               color=['#95A5A6', '#3498DB', '#27AE60', '#16A085'])
    ax.set_title(f'{metrica} por estrategia de mitigación', fontweight='bold')
    ax.set_ylabel(metrica); ax.set_ylim(0, max(0.8, pivot.max().max() * 1.25))
    ax.legend(fontsize=7, loc='upper left')
    for cont in ax.containers:
        ax.bar_label(cont, fmt='%.3f', fontsize=7, padding=2)
fig.suptitle('Ablación: sin mitigación vs. class_weight vs. umbralización',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figs/exp3_ablacion_desbalance.png', dpi=200, bbox_inches='tight')   # [Z.3]
plt.show()


### 6.2 Análisis Profundo de Errores — El Problema de la Clase Neutral


In [ ]:
# ─── Matrices de confusión comparativas ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (name, data) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, data['y_pred'], normalize='true')
    sns.heatmap(
        cm, annot=True, fmt='.2%', cmap='Blues', ax=ax,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.5, linecolor='gray', cbar=False,
        annot_kws={'size': 10, 'weight': 'bold'}
    )
    ax.set_title(f'{name}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

fig.suptitle('Matrices de Confusión Normalizadas — Conjunto de Test',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figs/exp3_confusion_matrices_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

# ─── Análisis de ejemplos mal clasificados ───────────────────────────────────
print("[Ejemplos mal clasificados por el mejor modelo neuronal]")
best_model_preds = y_pred_bilstm  # Reemplazar por el mejor según métricas

error_mask   = best_model_preds != y_test
X_test_texts = X_test[error_mask]
y_true_err   = y_test[error_mask]
y_pred_err   = best_model_preds[error_mask]

print(f"\nTotal de errores en test: {error_mask.sum()} / {len(y_test)} ({error_mask.mean()*100:.1f}%)\n")
print("-" * 80)

# Mostrar 5 ejemplos de errores involucrando la clase Neutral
neutral_errors = [(t, tr, pr) for t, tr, pr in zip(X_test_texts, y_true_err, y_pred_err)
                  if tr == 1 or pr == 1]
for i, (text, true, pred) in enumerate(neutral_errors[:5]):
    print(f"Ejemplo {i+1}:")
    print(f"  Texto    : {text[:120]}...")
    print(f"  Real     : {INV_LABEL[true]}   |   Predicho: {INV_LABEL[pred]}")
    print("-" * 80)

---
## Sección 7 — Reflexión Final y Recomendación para Producción

### 7.1 Síntesis de Hallazgos

| Dimensión | TF-IDF + RL | CNN-1D | BiLSTM |
|---|---|---|---|
| Captura de orden secuencial | ✗ | ✓ (local, n-gramas) | ✓✓ (global, largo alcance) |
| Manejo de negaciones | ✗ parcial | ✓ (trigramas) | ✓✓ |
| Robustez a errores ortográficos | media | alta (corpus propio) | alta (corpus propio) |
| Velocidad de inferencia | muy alta | alta | media |
| Costo computacional de entrenamiento | bajo | medio | alto |
| Interpretabilidad | alta | baja | baja |
| F1 clase Neutral | bajo | medio | medio-alto |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Serialización de métricas reportables + exportación de figuras
# ═══════════════════════════════════════════════════════════════════════════════
import json, os, platform, shutil, sklearn, matplotlib, seaborn
from pathlib import Path

Path('figs').mkdir(exist_ok=True)
T_FIN_GLOBAL = time.perf_counter()

M = {}   # dict PLANO: nombre -> valor

def _reg(clave, valor, dec=4):
    """Registra un valor; redondea floats a 4 decimales. No inventa claves."""
    if valor is None:
        M[clave] = None
    elif isinstance(valor, (float, np.floating)):
        M[clave] = round(float(valor), dec)
    elif isinstance(valor, (int, np.integer, bool)):
        M[clave] = int(valor)
    else:
        M[clave] = str(valor)

# ─── 1. Entorno, versiones y semilla ──────────────────────────────────────────
_reg('entorno_python',        platform.python_version())
_reg('entorno_plataforma',    platform.platform())
_reg('version_tensorflow',    tf.__version__)
_reg('version_numpy',         np.__version__)
_reg('version_pandas',        pd.__version__)
_reg('version_sklearn',       sklearn.__version__)
_reg('version_matplotlib',    matplotlib.__version__)
_reg('version_seaborn',       seaborn.__version__)
_reg('semilla',               SEED)
_reg('acelerador',            ACELERADOR)
_reg('tiempo_total_notebook_s', T_FIN_GLOBAL - T_INICIO_GLOBAL)

# ─── 2. Corpus y reconciliación del N ──────────────────────────────
_reg('corpus_n_original',        N_ORIGINAL)
_reg('corpus_n_nulos',           N_NULOS)
_reg('corpus_n_duplicados',      N_DUPS)
_reg('corpus_n_solapamiento',    N_SOLAPAMIENTO)
_reg('corpus_n_eliminadas_union', N_ELIMINADAS)
_reg('corpus_n_final',           N_FINAL)

# ─── 3. Particiones: N exacto y soporte por clase ─────────────────────────────
for _split, _y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    _reg(f'n_{_split}', len(_y))
    _c = Counter(_y)
    for _cl_nombre, _cl_idx in LABEL_MAP.items():
        _reg(f'n_{_split}_{_cl_nombre.lower()}', int(_c.get(_cl_idx, 0)))
        _reg(f'pct_{_split}_{_cl_nombre.lower()}', _c.get(_cl_idx, 0) / len(_y) * 100)

# ─── 4. Preprocesamiento ──────────────────────────────────────────────────────
_reg('vocab_size_config',     VOCAB_SIZE)
_reg('vocab_corpus_unico',    len(tokenizer.word_index))
_reg('max_len_eda_p95',       MAX_LEN_EDA)        # 68 esperado
_reg('max_len_aplicado_p95',  MAX_LEN_APLICADO)   # 63 esperado
_reg('max_len_diferencia',    MAX_LEN_EDA - MAX_LEN_APLICADO)
_reg('tfidf_max_features_config', 15000)
_reg('tfidf_features_reales', X_train_tfidf.shape[1])
for _k, _v in class_weight_dict.items():
    _reg(f'class_weight_{INV_LABEL[_k].lower()}', _v)

# ─── 5. Métricas por modelo (desde REPORTES) ────────────────────────
_SLUG = {'TF-IDF + Reg. Logística': 'tfidf_lr', 'BiLSTM': 'bilstm', 'CNN-1D': 'cnn1d'}
for _nombre, _rep in REPORTES.items():
    s = _SLUG[_nombre]
    _reg(f'{s}_accuracy',            _rep['accuracy'])
    _reg(f'{s}_macro_f1',            _rep['macro avg']['f1-score'])
    _reg(f'{s}_weighted_f1',         _rep['weighted avg']['f1-score'])
    for _cl in CLASS_NAMES:
        _reg(f'{s}_f1_{_cl.lower()}',        _rep[_cl]['f1-score'])
        _reg(f'{s}_precision_{_cl.lower()}', _rep[_cl]['precision'])
        _reg(f'{s}_recall_{_cl.lower()}',    _rep[_cl]['recall'])
    _reg(f'{s}_params_entrenables', PARAMS_MODELO[_nombre])

# ─── 6. Tiempos y latencias ────────────────────────────────────────
for _nombre, _t in TIEMPOS.items():
    s = _SLUG[_nombre]
    _reg(f'{s}_train_segundos',       _t['segundos_entrenamiento'])
    _reg(f'{s}_epocas_ejecutadas',    _t['epocas_ejecutadas'])
    _reg(f'{s}_epoca_mejor_val_loss', _t['epoca_mejor_val_loss'])
    _reg(f'{s}_segundos_por_epoca',   _t['segundos_por_epoca'])
    _reg(f'{s}_segundos_hasta_mejor', _t['segundos_hasta_mejor'])
    _reg(f'{s}_mejor_val_loss',       _t['mejor_val_loss'])
for _nombre, _l in LATENCIAS.items():
    s = _SLUG[_nombre]
    _reg(f'{s}_latencia_ms_lote64',       _l['ms_lote64'])
    _reg(f'{s}_latencia_ms_por_muestra',  _l['ms_por_muestra_b64'])
    _reg(f'{s}_latencia_ms_batch1',       _l['ms_batch1'])

# ─── 7. Umbralización ──────────────────────────────────────────────
for _nombre, _r in UMBRAL.items():
    s = _SLUG[_nombre]
    _reg(f'{s}_umbral_neutral_optimo',      _r['umbral_optimo'])
    _reg(f'{s}_macro_f1_antes_umbral',      _r['rep_antes']['macro avg']['f1-score'])
    _reg(f'{s}_macro_f1_despues_umbral',    _r['rep_despues']['macro avg']['f1-score'])
    _reg(f'{s}_f1_neutral_antes_umbral',    _r['rep_antes']['Neutral']['f1-score'])
    _reg(f'{s}_f1_neutral_despues_umbral',  _r['rep_despues']['Neutral']['f1-score'])
    _reg(f'{s}_recall_neutral_antes_umbral',   _r['rep_antes']['Neutral']['recall'])
    _reg(f'{s}_recall_neutral_despues_umbral', _r['rep_despues']['Neutral']['recall'])
    _reg(f'{s}_accuracy_antes_umbral',      _r['rep_antes']['accuracy'])
    _reg(f'{s}_accuracy_despues_umbral',    _r['rep_despues']['accuracy'])

# ─── 8. Ablación de estrategias ────────────────────────────────────
for _, _fila_ab in df_ablacion.iterrows():
    _pref = f"ablacion_{_SLUG[_fila_ab['Modelo']]}_{_fila_ab['Estrategia'].split(' ')[0].lower()}"
    _reg(f'{_pref}_accuracy',       _fila_ab['Accuracy'])
    _reg(f'{_pref}_macro_f1',       _fila_ab['Macro-F1'])
    _reg(f'{_pref}_f1_neutral',     _fila_ab['F1-Neutral'])
    _reg(f'{_pref}_recall_neutral', _fila_ab['Recall-Neutral'])
    _reg(f'{_pref}_n_pred_neutral', _fila_ab['Pred. Neutral (n)'])

# ─── 9. Escritura del JSON ────────────────────────────────────────────────────
with open('metricas_exp3.json', 'w', encoding='utf-8') as f:
    json.dump(M, f, ensure_ascii=False, indent=2, sort_keys=True)

print(f"✓ metricas_exp3.json escrito con {len(M)} claves.")
print("\n[Muestra de claves registradas]")
for k in sorted(M)[:25]:
    print(f"  {k:<42} = {M[k]}")
print(f"  ... ({len(M) - 25} claves más)")

# ─── 10. Reexportación de TODAS las figuras abiertas a figs/ @200 dpi ─────────
NOMBRES_FIG = {}    # num_figura -> nombre de archivo (completar según orden real)
_guardadas = []
for _num in plt.get_fignums():
    _f = plt.figure(_num)
    _sup = _f._suptitle.get_text() if _f._suptitle else f'figura_{_num}'
    _slug = re.sub(r'[^a-z0-9]+', '_', _sup.lower()).strip('_')[:60] or f'figura_{_num}'
    _ruta = f'figs/exp3_{_slug}.png'
    _f.savefig(_ruta, dpi=200, bbox_inches='tight')
    _guardadas.append(_ruta)

print(f"\n✓ {len(_guardadas)} figuras exportadas a figs/ @200 dpi:")
for _r in _guardadas:
    print(f"  {_r}")

# ─── 11. Empaquetado de entregables (portable: shutil, sin !zip) ──────────────
_stage = Path('_entregables_exp3_tmp')
if _stage.exists():
    shutil.rmtree(_stage)
_stage.mkdir()
shutil.copytree('figs', _stage / 'figs')
shutil.copy('metricas_exp3.json', _stage / 'metricas_exp3.json')
shutil.make_archive('entregables_exp3', 'zip', root_dir=str(_stage))
shutil.rmtree(_stage)
print("\n✓ entregables_exp3.zip listo (figs/ + metricas_exp3.json).")

# Descarga automática solo si se corre en Colab; fuera de Colab no rompe.
try:
    from google.colab import files
    files.download('entregables_exp3.zip')
except Exception as _e:
    print(f"  (descarga automática no disponible fuera de Colab: {_e})")
